最新、最简洁的API，能让模型理解我们需要的数据结构，并返回解析好的对象
还可以传入参数incluede_raw=True，默认是False 表示返回解析前的原始AIMessage,从而访问token用量等元数据

In [1]:
from dataclasses import dataclass

from pydantic import BaseModel, Field, ValidationError
from langchain_core.tools import tool
from langchain_core.utils.function_calling import convert_to_openai_tool
from rich import print as rprint
from typing import Literal, Optional
from langchain.chat_models import init_chat_model
from rich import print as rprint
##统一使用init_chat_model（）完成模型的初始化
import os
from dotenv import load_dotenv
from langchain_core.messages import SystemMessage, HumanMessage, AIMessage
from langchain_core.prompts import ChatPromptTemplate


#1 读取，env文件中的配置信息，相关的环境变量以.env文件中的优先
load_dotenv(override=True)
my_api_key=os.getenv("API_KEY")
my_base_url=os.getenv("BASE_URL")
#2 初始化大模型
model= init_chat_model(
    model="qwen3.8-max",
    model_provider="openai",
    api_key=my_api_key,
    base_url=my_base_url,
    timeout=30,
    max_retries=3,
)
message=[HumanMessage("我想知道胡歌的一些信息")]
@dataclass
class Person():
    """
    人物信息
    """
    name: str=Field(description="姓名",min_length=2,max_length=20)
    age:int=Field(description="年龄",gt=0,lt=120)
    occupation:str=Field(description="职业")
    hobby:str=Field(description="爱好")
    height:int=Field(description="身高",gt=0,lt=300)
    gender:str=Field(description="性别")
    weight:int=Field(description="体重",gt=0)
structured_model=model.with_structured_output(Person)

result=structured_model.invoke(message)
print(result)

{'name': '胡歌', 'age': 41, 'occupation': '演员', 'hobby': '表演、摄影', 'height': 185, 'gender': '男', 'weight': 75}


使用Include_raw参数的示例

In [3]:

from rich import print as rprint
structured_model=model.with_structured_output(Person,include_raw=True)
result=structured_model.invoke(message)
rprint(result)

{
    'raw': AIMessage(
        content='{\n  "name": "胡歌",\n  "age": 42,\n  "occupation": "演员、歌手",\n  "hobby": 
"阅读、旅行、摄影",\n  "height": 185,\n  "gender": "男",\n  "weight": 70\n}',
        additional_kwargs={'parsed': None, 'refusal': None},
        response_metadata={
            'token_usage': {
                'completion_tokens': 305,
                'prompt_tokens': 299,
                'total_tokens': 604,
                'completion_tokens_details': {
                    'accepted_prediction_tokens': None,
                    'audio_tokens': None,
                    'reasoning_tokens': 232,
                    'rejected_prediction_tokens': None,
                    'text_tokens': 305
                },
                'prompt_tokens_details': {
                    'audio_tokens': None,
                    'cache_write_tokens': None,
                    'cached_tokens': 0,
                    'image_tokens': None,
                    'text_tokens': 299
                }
            },
            'model_provider': 'openai',
            'model_name': 'qwen3.8-max',
            'system_fingerprint': None,
            'id': 'chatcmpl-fa0f13b1-87af-92fb-911f-990e554a1313',
            'finish_reason': 'stop',
            'logprobs': None
        },
        id='lc_run--01a0a578-bb85-7dc2-a3ff-e52611760f5a-0',
        tool_calls=[],
        invalid_tool_calls=[],
        usage_metadata={
            'input_tokens': 299,
            'output_tokens': 305,
            'total_tokens': 604,
            'input_token_details': {'cache_read': 0},
            'output_token_details': {'reasoning': 232}
        }
    ),
    'parsed': {
        'name': '胡歌',
        'age': 42,
        'occupation': '演员、歌手',
        'hobby': '阅读、旅行、摄影',
        'height': 185,
        'gender': '男',
        'weight': 70
    },
    'parsing_error': None
}